<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/paper_summary/03_SLCP_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 03 — Ratio corrections over both flow bases

This stage fits both correction factorizations over both pretrained bases.  For
exact JANA it uses the nominal JANA posterior and likelihood proposals, as the
reproduction flow is already expected to be accurate.  For the separate flows
selected in notebook 01, the learned transports are driven by a single
broadened Gaussian latent base; the posterior proposal adds a defensive prior
component, while the observation-space likelihood proposal uses broadening
alone.  No posterior or likelihood flow is retrained here.

Finish every selected budget/seed in notebook 02 first. This notebook reuses
its saved training contracts (including batch size 1024 for 1M), even when
TensorFlow evaluation runs on CPU. Missing or unfinished flows are reported
before exporting ratio banks; return to 02 with `LOAD_IF_AVAILABLE=True` to
finish them. Changing `LOAD_IF_AVAILABLE` here affects corrections and
diagnostics, not the pretrained flow weights.

At 100k, a disjoint train/checkpoint-validation/untouched-closure split and
importance efficiency select one `(tau, epsilon)` pair without access to
reference posterior samples.  The rule takes the worst route and worst of the
multiclass/binary factorizations, so proposal tuning is symmetric.  That pair
is then frozen across budgets.  One equal-prior three-class CE ensemble is
compared with two independent equal-prior binary CE ensembles for each base.
All genuine classifier examples come from the same training rows used by the
corresponding flows.  The 100k proposal ablation applies to the separate-flow
base and is retained as a control, not as another headline pipeline.


## Repairing saved-model evaluation (02 and 03)

The pinned BayesFlow version stores its orthogonal rotation matrices as ordinary
Tensors, so they are absent from TensorFlow checkpoints. The inference loader
now reconstructs these matrices with the saved **training seed** before restoring
the trained weights. Previously, fresh random rotations could make all posterior
proposals fall outside the prior and raise `All likelihood-route importance
weights are zero`, even though training had completed successfully.

Rerun this notebook from the setup cell with `LOAD_IF_AVAILABLE=True` and the
same artifact root. No completed flow needs retraining. Old JANA diagnostics
and ratio banks are preserved under recovery names and regenerated once.
JANA correction classifiers trained on the old banks must also be fitted again;
they are preserved separately from the corrected classifiers. Separate-flow
models and their corrections are unaffected. Subsequent runs reuse the repaired
outputs normally. The prior, proposals and importance-weight formula are unchanged.


In [1]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import importlib.util
import json
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

# Must be set before the first CUDA/PyTorch initialization in this process.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = os.environ.get("PAPER_SUMMARY_USE_DRIVE", "1") != "0"

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

def installed_version(distribution):
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return None

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        default_artifact_root = Path(
            "/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP"
        )
    else:
        default_artifact_root = Path("/content/paper_summary_SLCP_artifacts")

    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, repository, env=clone_env,
        )
    else:
        run("git", "-C", repository, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", repository, "fetch", "origin", BRANCH)
        run("git", "-C", repository, "checkout", BRANCH)
        run("git", "-C", repository, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", repository, "sparse-checkout", "set", "src",
        "workshops/ml4hep_tifr_colab/paper_summary",
    )
    SOURCE_DIR = repository / "workshops" / "ml4hep_tifr_colab" / "paper_summary"

    # Colab already provides the numerical/ML stack used by these notebooks.
    # Install only the two missing modern-runtime packages normally.  In
    # particular, do not let sbibm pull its historical algorithm dependency
    # tree into the current Colab Python environment (currently Python 3.13).
    modern_requirements = []
    if installed_version("nflows") != "0.14":
        modern_requirements.append("nflows==0.14")
    if importlib.util.find_spec("pyro") is None:
        modern_requirements.append("pyro-ppl")
    if modern_requirements:
        run(sys.executable, "-m", "pip", "install", "-q", *modern_requirements)
    if installed_version("sbibm") != "1.1.0":
        # This is the same Python-3.13-safe installation used by Exercises 9
        # and 10: the SLCP task/metrics need sbibm itself, nflows, and Pyro,
        # but not sbibm's old pinned SBI/algorithm environment.
        run(
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            "sbibm==1.1.0",
        )
else:
    candidates = (
        Path.cwd(),
        Path.cwd() / "paper_summary",
        Path.cwd() / "workshops" / "ml4hep_tifr_colab" / "paper_summary",
    )
    SOURCE_DIR = next(
        (candidate.resolve() for candidate in candidates if (candidate / "config.py").is_file()),
        None,
    )
    if SOURCE_DIR is None:
        raise FileNotFoundError("Cannot locate the paper_summary source directory")
    default_artifact_root = SOURCE_DIR / "artifacts"

source_path = str(SOURCE_DIR)
if source_path not in sys.path:
    sys.path.insert(0, source_path)
os.chdir(SOURCE_DIR)

ARTIFACT_ROOT = Path(
    os.environ.get("PAPER_SUMMARY_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("Paper-summary source:", SOURCE_DIR)
print("Persistent artifact root:", ARTIFACT_ROOT)


Mounted at /content/drive
Paper-summary source: /content/nsbi-lhc-toolkit/workshops/ml4hep_tifr_colab/paper_summary
Persistent artifact root: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP


In [2]:
from config import (
    DEFAULT_ML_SEEDS,
    PAPER_BUDGETS,
    SMOKE_BUDGETS,
    SMOKE_ML_SEEDS,
    campaign_config,
    campaign_signature,
)

PROFILE = os.environ.get("PAPER_SUMMARY_PROFILE", "PAPER").upper()
CAMPAIGN_BUDGETS = list(PAPER_BUDGETS if PROFILE == "PAPER" else SMOKE_BUDGETS)
CAMPAIGN_ML_SEEDS = list(DEFAULT_ML_SEEDS if PROFILE == "PAPER" else SMOKE_ML_SEEDS)

def execution_subset(environment_name, configured):
    raw = os.environ.get(environment_name, "").strip()
    values = list(configured) if not raw else [int(value) for value in raw.split(",")]
    unknown = set(values) - set(configured)
    if not values or unknown:
        raise ValueError(f"Invalid {environment_name}: {values}; unknown={sorted(unknown)}")
    return values

BUDGETS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_BUDGETS", CAMPAIGN_BUDGETS)
ML_SEEDS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_SEEDS", CAMPAIGN_ML_SEEDS)
LOAD_IF_AVAILABLE = os.environ.get("PAPER_SUMMARY_LOAD_IF_AVAILABLE", "1") != "0"

CAMPAIGN = campaign_config(profile=PROFILE)
print(json.dumps({
    "profile": PROFILE,
    "campaign_budgets": CAMPAIGN_BUDGETS,
    "campaign_ml_seeds": CAMPAIGN_ML_SEEDS,
    "budgets_to_run": BUDGETS_TO_RUN,
    "ml_seeds_to_run": ML_SEEDS_TO_RUN,
    "load_if_available": LOAD_IF_AVAILABLE,
    "campaign_signature": campaign_signature(CAMPAIGN),
}, indent=2))


{
  "profile": "PAPER",
  "campaign_budgets": [
    10000,
    100000,
    1000000
  ],
  "campaign_ml_seeds": [
    31082026,
    31082027,
    31082028
  ],
  "budgets_to_run": [
    10000,
    100000,
    1000000
  ],
  "ml_seeds_to_run": [
    31082026,
    31082027,
    31082028
  ],
  "load_if_available": true,
  "campaign_signature": "sha256-e455fa167513"
}


In [3]:
FACTORIZATIONS = ("multiclass", "binary")
RUN_PROPOSAL_ABLATION = True
INSTALL_EXACT_JANA_ENV_IF_MISSING = (
    os.environ.get("PAPER_SUMMARY_INSTALL_JANA_ENV", "1") != "0"
)


In [4]:
# Reload so rerunning this cell after the setup cell pulls a repository
# update cannot retain an older helper from the current Colab process.
import importlib
import utils_jana
import utils_jana_runtime
import utils_jana_gpu
import utils_jana_reuse
import utils_jana_checkpoint

utils_jana = importlib.reload(utils_jana)
utils_jana_runtime = importlib.reload(utils_jana_runtime)
utils_jana_gpu = importlib.reload(utils_jana_gpu)
utils_jana_reuse = importlib.reload(utils_jana_reuse)
utils_jana_checkpoint = importlib.reload(utils_jana_checkpoint)

print("Preparing the isolated exact-JANA runtime (first install can take several minutes).")
JANA_PYTHON = utils_jana_runtime.ensure_jana_environment(
    ARTIFACT_ROOT,
    install_if_missing=INSTALL_EXACT_JANA_ENV_IF_MISSING,
)
print("Exact-JANA Python:", JANA_PYTHON)


Preparing the isolated exact-JANA runtime (first install can take several minutes).
Rebuilding isolated exact-JANA environment: /content/paper_summary_jana_env
Installing pinned exact-JANA packages into: /content/paper_summary_jana_env
Exact-JANA installation probe: {"base_prefix": "/root/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu", "executable": "/content/paper_summary_jana_env/bin/python", "numpy": "1.23.5", "prefix": "/content/paper_summary_jana_env", "purelib": "/content/paper_summary_jana_env/lib/python3.11/site-packages", "python": "3.11.16", "site_packages": ["/content/paper_summary_jana_env/lib/python3.11/site-packages"]}
Exact-JANA Python: /content/paper_summary_jana_env/bin/python


In [5]:
from IPython.display import display

def display_result(result):
    if hasattr(result, "style"):
        display(result.style.format(precision=4).hide(axis="index"))
    elif isinstance(result, dict):
        for name, value in result.items():
            print(f"\n{name}")
            if hasattr(value, "style"):
                display(value.style.format(precision=4).hide(axis="index"))
            else:
                display(value)
    else:
        display(result)


In [6]:
import importlib
import utils

# Pull repository fixes into an already-open Colab runtime.
utils = importlib.reload(utils)

HYBRID_RESULT = utils.run_hybrid_campaign(
    artifact_root=ARTIFACT_ROOT,
    campaign=CAMPAIGN,
    factorizations=FACTORIZATIONS,
    run_proposal_ablation=RUN_PROPOSAL_ABLATION,
    budgets_to_run=BUDGETS_TO_RUN,
    ml_seeds_to_run=ML_SEEDS_TO_RUN,
    load_if_available=LOAD_IF_AVAILABLE,
)
display_result(HYBRID_RESULT)


[exact JANA reuse] budget_n0010000/seed_31082026: completed checkpoint, batch size 32; no flow training.
[exact JANA reuse] budget_n0010000/seed_31082027: completed checkpoint, batch size 32; no flow training.
[exact JANA reuse] budget_n0010000/seed_31082028: completed checkpoint, batch size 32; no flow training.
/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/jana_paper/campaign_manifest.json
/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/ratio_banks/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/manifest.json
/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/ratio_banks/jana_paper/sha256-e455fa167513/budget_10000/seed_31082027/manifest.json
/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/ratio_banks/jana_paper/sha256-e455fa167513/budget_10000/seed_31082028/manifest.json
[exact JANA reuse] budget_n0100000/seed_31082026: completed checkpoint, batch size 32; no flow training.
[exact JANA reuse] budget_n0100000/seed_31082027: completed checkp

/usr/local/lib/python3.13/dist-packages/nflows/transforms/lu.py:80: UserWarning: torch.triangular_solve is deprecated in favor of torch.linalg.solve_triangularand will be removed in a future PyTorch release.
torch.linalg.solve_triangular has its arguments reversed and does not return a copy of one of the inputs.
X = torch.triangular_solve(B, A).solution
should be replaced with
X = torch.linalg.solve_triangular(A, B). (Triggered internally at /pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp:2259.)
  outputs, _ = torch.triangular_solve(


Flow-mixture member 1/4: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/matched/sha256-e455fa167513/budget_1000000/seed_31082026/q_phi.member_00.pt
Loaded spline flow with matching data fingerprint from /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/matched/sha256-e455fa167513/budget_1000000/seed_31082026/q_phi.member_00.pt
Flow-mixture member 2/4: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/matched/sha256-e455fa167513/budget_1000000/seed_31082026/q_phi.member_01.pt
Loaded spline flow with matching data fingerprint from /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/matched/sha256-e455fa167513/budget_1000000/seed_31082026/q_phi.member_01.pt
Flow-mixture member 3/4: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/models/matched/sha256-e455fa167513/budget_1000000/seed_31082026/q_phi.member_02.pt
Loaded spline flow with matching data fingerprint from /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/mod

{'campaign_signature': 'sha256-e455fa167513',
 'closure_mean': 0.3176119923591614,
 'closure_threshold': 0.33223435210215735,
 'created_utc': '2026-09-11T19:23:32.181324+00:00',
 'frozen_across_budgets': True,
 'minimum_ESS_mean': 0.014272847093076628,
 'pilot_split': 'validation_partition_split_once_into_disjoint_train_checkpoint_validation_and_untouched_closure',
 'proposal_base_scale': 1.25,
 'proposal_prior_fraction': 0.05,
 'ranking': 'worst_route_and_factorization_within_one_uncertainty_of_best_joint_closure_then_highest_worst_factorization_minimum_ESS',
 'schema': 'slcp_paper_summary_v2',
 'selection_budget': 100000,
 'selection_budget_is_smoke_substitute': False,
 'selection_budget_requested': 100000,
 'selection_factorizations': ['binary', 'multiclass'],
 'uses_audit_bank': False,
 'uses_reference_posterior': False}


proposal_scan


...

schema,campaign_signature,method,factorization,budget,simulator_calls,training_rows,validation_rows,ml_seed,observation,posterior_C2ST,posterior_MMD,likelihood_posterior_C2ST,likelihood_posterior_MMD,posterior_likelihood_route_C2ST,posterior_likelihood_route_MMD,posterior_ESS_fraction,posterior_max_weight,likelihood_posterior_ESS_fraction,likelihood_posterior_max_weight,bayes_cycle_pearson,bayes_cycle_slope,bayes_cycle_residual_rms,bayes_cycle_rows,bayes_cycle_theta_fingerprint,likelihood_log_Z_rms,likelihood_log_Z_mean,likelihood_log_Z_max_abs,exact_likelihood_log_error,exact_likelihood_centered_log_error,exact_likelihood_rows,likelihood_audit_rows,likelihood_audit_fingerprint,audit_bank_fingerprint,deployed_proposal_exact_likelihood_log_error,deployed_proposal_exact_likelihood_centered_log_error,deployed_proposal_exact_likelihood_rows,deployed_proposal_bayes_cycle_pearson,deployed_proposal_bayes_cycle_slope,deployed_proposal_bayes_cycle_residual_rms,deployed_proposal_bayes_cycle_rows,proposal_base_scale,proposal_prior_fraction,posterior_ratio_member_log_std,likelihood_ratio_member_log_std,ratio_selected_validation_CE_mean,ratio_selected_validation_CE_std,posterior_joint_C2ST,posterior_joint_MMD,predictive_x_C2ST,predictive_x_MMD,predictive_joint_C2ST,predictive_joint_MMD,posterior_joint_ESS_fraction,likelihood_joint_ESS_fraction,inference_revision
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,1,0.9209,0.5332,0.7718,0.1705,0.8663,0.2814,0.7646,0.0000,0.0045,0.0280,0.5181,7.2279,208.1050,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,103418222163239072.0000,103417687210606752.0000,150000,0.2908,9.3167,44.7446,137662,1.0000,0.0000,0.1772,0.1737,1.0936,0.0028,0.5533,0.0213,0.5315,0.0202,0.5316,0.0168,0.7926,0.9666,seeded_orthogonal_restore_v1
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,2,0.8810,0.3968,0.6951,0.1040,0.8976,0.2919,0.6825,0.0000,0.0059,0.0166,0.6153,8.0595,450.7037,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,47516773954738700288.0000,47516614972969246720.0000,150000,0.2057,4.6268,28.8570,110173,1.0000,0.0000,0.2994,0.1872,1.0936,0.0028,0.5533,0.0213,0.5315,0.0202,0.5316,0.0168,0.7926,0.9666,seeded_orthogonal_restore_v1
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,3,0.8046,0.2669,0.7675,0.3324,0.8563,0.2826,0.5181,0.0001,0.0048,0.0196,0.5568,8.1210,328.3758,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,33199211240457208.0000,33199098697914288.0000,150000,0.2271,2.0519,11.3973,103739,1.0000,0.0000,0.2428,0.1330,1.0936,0.0028,0.5533,0.0213,0.5315,0.0202,0.5316,0.0168,0.7926,0.9666,seeded_orthogonal_restore_v1
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,4,0.9209,0.4276,0.8402,0.1250,0.8628,0.1745,0.9216,0.0000,0.0154,0.0160,0.2752,0.4027,77.9117,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,1733839150359023616.0000,1733833334687485696.0000,150000,0.4935,2.2348,7.7200,143324,1.0000,0.


separate_flow_corrections


schema,campaign_signature,method,factorization,budget,simulator_calls,training_rows,validation_rows,ml_seed,observation,posterior_C2ST,posterior_MMD,likelihood_posterior_C2ST,likelihood_posterior_MMD,posterior_likelihood_route_C2ST,posterior_likelihood_route_MMD,posterior_ESS_fraction,posterior_max_weight,likelihood_posterior_ESS_fraction,likelihood_posterior_max_weight,bayes_cycle_pearson,bayes_cycle_slope,bayes_cycle_residual_rms,bayes_cycle_rows,bayes_cycle_theta_fingerprint,likelihood_log_Z_rms,likelihood_log_Z_mean,likelihood_log_Z_max_abs,exact_likelihood_log_error,exact_likelihood_centered_log_error,exact_likelihood_rows,likelihood_audit_rows,likelihood_audit_fingerprint,audit_bank_fingerprint,deployed_proposal_exact_likelihood_log_error,deployed_proposal_exact_likelihood_centered_log_error,deployed_proposal_exact_likelihood_rows,deployed_proposal_bayes_cycle_pearson,deployed_proposal_bayes_cycle_slope,deployed_proposal_bayes_cycle_residual_rms,deployed_proposal_bayes_cycle_rows,proposal_base_scale,proposal_prior_fraction,posterior_ratio_member_log_std,likelihood_ratio_member_log_std,ratio_selected_validation_CE_mean,ratio_selected_validation_CE_std,posterior_joint_C2ST,posterior_joint_MMD,predictive_x_C2ST,predictive_x_MMD,predictive_joint_C2ST,predictive_joint_MMD,posterior_joint_ESS_fraction,likelihood_joint_ESS_fraction
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows_corrected_multiclass,multiclass,10000,10000,8999,1001,31082026,1,0.9690,0.5889,0.9573,0.5312,0.6381,0.1327,0.0245,0.0067,0.0321,0.0012,0.9397,1.1117,1.3939,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,1.4898,0.7784,6.0086,585900251.3243,585869257.4684,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,978752328592816865280.0000,978748296356677943296.0000,150000,0.9270,1.0924,1.6455,150000,1.2500,0.0500,1.5961,1.4222,0.5806,0.0092,0.8854,0.2182,0.9282,0.4021,0.9317,0.3363,0.0067,0.0010
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows_corrected_multiclass,multiclass,10000,10000,8999,1001,31082026,2,0.9552,0.5150,0.9541,0.4192,0.6656,0.1590,0.0166,0.0023,0.0044,0.0085,0.9572,0.9683,1.1157,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,1.4898,0.7784,6.0086,585900251.3243,585869257.4684,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,203514086462802853101568.0000,203513407957016963448832.0000,150000,0.9599,0.9780,1.1895,150000,1.2500,0.0500,1.5903,2.0670,0.5806,0.0092,0.8854,0.2182,0.9282,0.4021,0.9317,0.3363,0.0067,0.0010
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows_corrected_multiclass,multiclass,10000,10000,8999,1001,31082026,3,0.9313,0.4919,0.8797,0.3586,0.7210,0.2447,0.0307,0.0015,0.0199,0.0037,0.9641,1.0633,1.4835,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,1.4898,0.7784,6.0086,585900251.3243,585869257.4684,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,298115087283594199040.0000,298112938997379760128.0000,150000,0.9690,0.9517,1.3897,150000,1.2500,0.0500,1.6520,1.5150,0.5806,0.0092,0.8854,0.2182,0.9282,0.4021,0.9317,0.3363,0.0067,0.0010
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows_corrected_multiclass,multiclass,10000,10000,8999,1001,31082026,4,0.9784,0.6861,0.9830,0.6963,0.7032,0.1025,0.0251,0.0018,0.0011,0.0444,0.9418,1.2512,1.8336,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,1.4898,0.7784,6.0086,585900251.3243,585869257.4684,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,19132365021001564160.0000,19132284995090370560.0000,150000,0.9154,1.2950,2.2080,150000,1.2500,0.0500,1.2035,0.9357,0.5806,0.0092,0.8854,0.2182,0.9282,0.4021,0.9317,0.3363,0


metrics


schema,campaign_signature,method,factorization,budget,simulator_calls,training_rows,validation_rows,ml_seed,observation,posterior_C2ST,posterior_MMD,likelihood_posterior_C2ST,likelihood_posterior_MMD,posterior_likelihood_route_C2ST,posterior_likelihood_route_MMD,posterior_ESS_fraction,posterior_max_weight,likelihood_posterior_ESS_fraction,likelihood_posterior_max_weight,bayes_cycle_pearson,bayes_cycle_slope,bayes_cycle_residual_rms,bayes_cycle_rows,bayes_cycle_theta_fingerprint,likelihood_log_Z_rms,likelihood_log_Z_mean,likelihood_log_Z_max_abs,exact_likelihood_log_error,exact_likelihood_centered_log_error,exact_likelihood_rows,likelihood_audit_rows,likelihood_audit_fingerprint,audit_bank_fingerprint,deployed_proposal_exact_likelihood_log_error,deployed_proposal_exact_likelihood_centered_log_error,deployed_proposal_exact_likelihood_rows,deployed_proposal_bayes_cycle_pearson,deployed_proposal_bayes_cycle_slope,deployed_proposal_bayes_cycle_residual_rms,deployed_proposal_bayes_cycle_rows,proposal_base_scale,proposal_prior_fraction,posterior_ratio_member_log_std,likelihood_ratio_member_log_std,ratio_selected_validation_CE_mean,ratio_selected_validation_CE_std,posterior_joint_C2ST,posterior_joint_MMD,predictive_x_C2ST,predictive_x_MMD,predictive_joint_C2ST,predictive_joint_MMD,posterior_joint_ESS_fraction,likelihood_joint_ESS_fraction,inference_revision
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,1,0.9209,0.5332,0.7718,0.1705,0.8663,0.2814,0.7646,0.0000,0.0045,0.0280,0.5181,7.2279,208.1050,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,103418222163239072.0000,103417687210606752.0000,150000,0.2908,9.3167,44.7446,137662,1.0000,0.0000,0.1772,0.1737,1.0936,0.0028,0.5533,0.0213,0.5315,0.0202,0.5316,0.0168,0.7926,0.9666,seeded_orthogonal_restore_v1
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,2,0.8810,0.3968,0.6951,0.1040,0.8976,0.2919,0.6825,0.0000,0.0059,0.0166,0.6153,8.0595,450.7037,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,47516773954738700288.0000,47516614972969246720.0000,150000,0.2057,4.6268,28.8570,110173,1.0000,0.0000,0.2994,0.1872,1.0936,0.0028,0.5533,0.0213,0.5315,0.0202,0.5316,0.0168,0.7926,0.9666,seeded_orthogonal_restore_v1
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,3,0.8046,0.2669,0.7675,0.3324,0.8563,0.2826,0.5181,0.0001,0.0048,0.0196,0.5568,8.1210,328.3758,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,33199211240457208.0000,33199098697914288.0000,150000,0.2271,2.0519,11.3973,103739,1.0000,0.0000,0.2428,0.1330,1.0936,0.0028,0.5533,0.0213,0.5315,0.0202,0.5316,0.0168,0.7926,0.9666,seeded_orthogonal_restore_v1
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper_corrected_multiclass,multiclass,10000,10304,10000,300,31082026,4,0.9209,0.4276,0.8402,0.1250,0.8628,0.1745,0.9216,0.0000,0.0154,0.0160,0.2752,0.4027,77.9117,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0478,0.0235,0.1179,585900251.4332,585869257.5308,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,1733839150359023616.0000,1733833334687485696.0000,150000,0.4935,2.2348,7.7200,143324,1.0000,0.